## Preprocesamiento de infraestructura territorial (OSM, DPA 2023, Censo 2024)

### Configuraciones iniciales

In [ ]:
# ── Configuración del repositorio ──────────────────────────────────────────
# Este cuaderno construye la infraestructura territorial del pipeline:
# Output Areas, teselación y los 18 atributos por unidad derivados de
# OpenStreetMap, más la población censal.
#
# Nivel 1 del repositorio: opera exclusivamente sobre fuentes públicas
# (OSM, DPA 2023 y Censo 2024) y es reproducible sin restricciones de acceso.
#
# Los cuadernos de este repositorio se publican sin resultados de ejecución.
# Lo que se entrega es el procedimiento; los valores de la corrida documentada
# figuran en el manuscrito y en el informe de montaje.

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import config as cfg

print(cfg.describe_config())

In [ ]:
# Importando librerías útiles
import gc
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from geopandas import sjoin
from pyogrio import list_layers, read_info
from pyproj import Geod
from shapely.geometry import box
from shapely.ops import unary_union
from IPython.display import display

### Lectura archivo .pbf y selección de registros de interés (según [query osm_query.yaml de Deep Gravity](https://github.com/scikit-mobility/DeepGravity/blob/master/osm_query.yaml))

In [ ]:
# Silencia únicamente el warning de anillos no cerrados en capas poligonales
warnings.filterwarnings("ignore", message="Non closed ring detected.*", category=RuntimeWarning)

pbf_path = cfg.require(cfg.OSM_PBF, "el extracto OSM de Chile (chile.osm.pbf)", nivel=1)

# 1) Listar capas
layers = list_layers(pbf_path)
for name, gtype in layers:
    info = read_info(pbf_path, layer=name, force_feature_count=True)
    print(f"{name:18s} | geom: {str(gtype):16s} | features: {info.get('features')} | crs: {info.get('crs')}")

# 2) Leer TODAS las capas sin seleccionar columnas
gdfs = {}
for name, _ in layers:
    gdf = gpd.read_file(pbf_path, engine="pyogrio", layer=name)

    # --- Corrección de anillos no cerrados / geometrías inválidas (solo capas poligonales) ---
    # Shapely >= 2: .make_valid() ; en Shapely 1.x, fallback con buffer(0)
    if name in ("multipolygons", "other_relations"):
        if hasattr(gdf.geometry, "make_valid"):
            gdf["geometry"] = gdf.geometry.make_valid()
        else:
            gdf["geometry"] = gdf.buffer(0)
        gdf = gdf[gdf.geometry.notna() & gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()

    gdfs[name] = gdf
    print(f"Capa '{name}': {len(gdf):,} filas, columnas: {list(gdf.columns)}")

# Accesos cómodos
points           = gdfs.get("points")
lines            = gdfs.get("lines")
multilinestrings = gdfs.get("multilinestrings")
multipolygons    = gdfs.get("multipolygons")
other_relations  = gdfs.get("other_relations")

# OPTIMIZACIÓN: Liberar variables temporales que ya no se necesitan
del name, gtype, info, gdf
_ = gc.collect()

Las tres secciones siguientes replican las consultas definidas en
`osm_query.yaml` del repositorio oficial de Deep Gravity, adaptadas a la
lectura del extracto nacional mediante `pyogrio`. La selección de entidades
(cláusula `where`) y de atributos (cláusula `select`) se conserva para
mantener comparabilidad con la especificación original de los atributos
territoriales.

#### Points

In [ ]:
points.head()

Ejecutamos querys de interés.

In [ ]:
# ---------- helpers ligeros para extraer desde other_tags ----------
def ensure_tag_col(gdf, key, col_name=None):
    """
    Si la columna `key` ya existe, la usa.
    De lo contrario, crea `col_name` extrayendo de `other_tags` sólo donde aparece `"key"=>`.
    Devuelve el nombre de columna utilizable (p.ej., 'amenity' o 'amenity' creado).
    """
    col = col_name or key.replace(":", "__")  # ej 'operator:type' -> 'operator__type'
    if key in gdf.columns:          # ya viene como columna "promovida"
        return key
    if col in gdf.columns:          # ya la habíamos creado
        return col

    gdf[col] = pd.NA
    if "other_tags" in gdf.columns:
        mask = gdf["other_tags"].notna() & gdf["other_tags"].str.contains(f'"{key}"=>', regex=False)
        if mask.any():
            # extrae valor "key"=>"valor"
            pattern = re.compile(rf'"{re.escape(key)}"=>"([^"]*)"')
            gdf.loc[mask, col] = gdf.loc[mask, "other_tags"].str.extract(pattern, expand=False)
    return col

# ---------- asegurar columnas necesarias (minimizando trabajo) ----------
# Claves usadas en el WHERE
keys_needed = [
    "building","shop","tourism","amenity","office","emergency","healthcare",
    "landuse","leisure","historic","sport","aeroway"
]
# Algunas pueden ya venir como columnas; para las que no, se crean desde other_tags
cols = {k: ensure_tag_col(points, k) for k in keys_needed}

# Aliases cortos
bld   = cols["building"]
shop  = cols["shop"]
tour  = cols["tourism"]
amen  = cols["amenity"]
off   = cols["office"]
emer  = cols["emergency"]
hcare = cols["healthcare"]
luse  = cols["landuse"]
leis  = cols["leisure"]
hist  = cols["historic"]
spt   = cols["sport"]
aero  = cols["aeroway"]

# ---------- máscaras que replican el YAML (planet_osm_point → where) ----------
m = (
    points[bld].notna() |
    points[shop].notna() | points[tour].notna() | points[amen].isin(["marketplace","restaurant","fast_food","cafe","bar","pub"]) | points[off].notna() |
    points[amen].isin(["kindergarten","school","college","university","language_school"]) | (points[off] == "educational_institution") |
    points[emer].notna() | points[amen].isin(["police","fire_station"]) |
    points[hcare].notna() | points[amen].isin(["doctors","dentist","clinic","toilets","hospital","pharmacy"]) | points[shop].isin(["herbalist","nutrition_supplements"]) |
    points[luse].notna() | (points[leis] == "park") | (points[amen] == "grave_yard") |
    (points[amen] == "fuel") |
    (points[amen] == "place_of_worship") |
    (points[amen] == "community_centre") |
    (points[amen] == "library") |
    points[hist].notna() |
    points[spt].notna() | points[leis].isin(["stadium","swimming_pool","pitch","sport_centre"]) |
    points[aero].notna() | (points[bld] == "aerodrome") |
    (points[amen] == "bus_station")
)

points_filt = points.loc[m].copy()

# ---------- columnas seleccionadas (replica del 'select' del YAML) ----------
# Nota: si alguna no existía, se crea desde other_tags sólo si la necesitas luego.
select_keys = [
    "access","aeroway","amenity","beds","building","capacity","denomination","emergency","fuel",
    "health_facility:bed","health_facility:level","health_facility:type","healthcare","historic",
    "isced:level","landuse","leisure","medical_system:western","name","opening_hours","operator",
    "operator:type","public_transport","religion","rooms","shop","staff_count:doctors",
    "staff_count:nurses","status","toilets:disposal","toilets:handwashing","tourism"
]
# Asegura como columnas sólo si vas a guardarlas
sel_map = {k: ensure_tag_col(points_filt, k) for k in select_keys}

# Arma el DataFrame final con selección
cols_out = ["osm_id"] + list(sel_map.values()) + ["geometry"]
points_out = points_filt[cols_out]

print(f"Filtrados POINTS: {len(points_out):,} filas (de {len(points):,})")
points_out.head()

# OPTIMIZACIÓN: Liberar variables temporales que ya no se necesitan
del keys_needed, cols, bld, shop, tour, amen, off, emer, hcare, luse, leis, hist, spt, aero, m, points_filt, select_keys, sel_map, cols_out
_ = gc.collect()

In [ ]:
print(list(points_out.columns))

In [ ]:
print(points_out.notna().sum().sort_values(ascending=False))

In [ ]:
# points_out.to_csv('points_out.csv', index=False)  # Guarda sin la columna de índice

#### Polygons

In [ ]:
multipolygons.head()

Ejecutamos querys de interés.

In [ ]:
# -- helper: crea columna desde other_tags sólo si no existe promovida --
if 'ensure_tag_col' not in globals():
    def ensure_tag_col(gdf, key, col_name=None):
        col = col_name or key.replace(":", "__")  # 'operator:type' -> 'operator__type'
        if key in gdf.columns:      # ya viene como columna propia
            return key
        if col in gdf.columns:      # ya lo habíamos creado
            return col
        gdf[col] = pd.NA
        if "other_tags" in gdf.columns:
            mask = gdf["other_tags"].notna() & gdf["other_tags"].str.contains(f'"{key}"=>', regex=False)
            if mask.any():
                pat = re.compile(rf'"{re.escape(key)}"=>"([^"]*)"')
                gdf.loc[mask, col] = gdf.loc[mask, "other_tags"].str.extract(pat, expand=False)
        return col

# --------- 1) columnas mínimas para construir el WHERE ----------
where_keys = [
    "building","shop","tourism","amenity","office","emergency","healthcare",
    "landuse","leisure","historic","sport","aeroway"
]
wcols = {k: ensure_tag_col(multipolygons, k) for k in where_keys}

bld   = wcols["building"]; shop  = wcols["shop"];   tour = wcols["tourism"]
amen  = wcols["amenity"];  off   = wcols["office"]; emer = wcols["emergency"]
hcare = wcols["healthcare"]; luse = wcols["landuse"]; leis = wcols["leisure"]
hist  = wcols["historic"];  spt  = wcols["sport"];  aero = wcols["aeroway"]

# --------- 2) máscara que replica el WHERE del YAML ----------
m = (
    multipolygons[bld].notna() |
    multipolygons[shop].notna() | multipolygons[tour].notna() |
    multipolygons[amen].isin(["marketplace","restaurant","fast_food","cafe","bar","pub"]) | multipolygons[off].notna() |
    multipolygons[amen].isin(["kindergarten","school","college","university","language_school"]) | (multipolygons[off]=="educational_institution") |
    multipolygons[emer].notna() | multipolygons[amen].isin(["police","fire_station"]) |
    multipolygons[hcare].notna() | multipolygons[amen].isin(["doctors","dentist","clinic","toilets","hospital","pharmacy"]) |
    multipolygons[shop].isin(["herbalist","nutrition_supplements"]) |
    multipolygons[luse].notna() | (multipolygons[leis]=="park") | (multipolygons[amen]=="grave_yard") |
    (multipolygons[amen]=="fuel") |
    (multipolygons[amen]=="place_of_worship") |
    (multipolygons[amen]=="community_centre") |
    (multipolygons[amen]=="library") |
    multipolygons[hist].notna() |
    multipolygons[spt].notna() | multipolygons[leis].isin(["stadium","swimming_pool","pitch","sport_centre"]) |
    multipolygons[aero].notna() | (multipolygons[bld]=="aerodrome") |
    (multipolygons[bld]=="train_station") |
    (multipolygons[amen]=="bus_station")
)

multipolygons_filt = multipolygons.loc[m].copy()
print(f"POLYGONS filtrados: {len(multipolygons_filt):,} / {len(multipolygons):,}")

# --------- 3) crear sólo ahora las columnas del SELECT (ahorra RAM) ----------
select_keys = [
    "access","aeroway","amenity","beds","building","capacity","denomination","emergency","fuel",
    "health_facility:bed","health_facility:level","health_facility:type","healthcare","historic",
    "isced:level","landuse","layer","leisure","medical_system:western","name","opening_hours",
    "operator","operator:type","public_transport","railway","religion","rooms","shop",
    "staff_count:doctors","staff_count:nurses","status","toilets:disposal","toilets:handwashing","tourism"
]
selmap = {k: ensure_tag_col(multipolygons_filt, k) for k in select_keys}

cols_out = ["osm_id"] + list(selmap.values()) + ["geometry"]
multipolygons_out = multipolygons_filt[cols_out]

multipolygons_out.head()

# OPTIMIZACIÓN: Liberar variables temporales que ya no se necesitan
del where_keys, wcols, bld, shop, tour, amen, off, emer, hcare, luse, leis, hist, spt, aero, m, multipolygons_filt, select_keys, selmap, cols_out
_ = gc.collect()

In [ ]:
print(list(multipolygons_out.columns))

In [ ]:
print(multipolygons_out.notna().sum().sort_values(ascending=False))

In [ ]:
# polygons_out.to_csv('polygons_out.csv', index=False)  # Guarda sin la columna de índice

#### lines

In [ ]:
lines.head()

In [ ]:
multilinestrings.head()

Ejecutamos querys de interés.

In [ ]:
# --- helper idempotente: saca una clave de `other_tags` si no existe como columna ---
if 'ensure_tag_col' not in globals():
    def ensure_tag_col(gdf, key, col_name=None):
        col = col_name or key.replace(":", "__")  # p.ej. 'operator:type' -> 'operator__type'
        if key in gdf.columns:              # ya viene promovida
            return key
        if col in gdf.columns:              # ya la habíamos creado
            return col
        gdf[col] = pd.NA
        if "other_tags" in gdf.columns:
            mask = gdf["other_tags"].notna() & gdf["other_tags"].str.contains(f'"{key}"=>', regex=False)
            if mask.any():
                pat = re.compile(rf'"{re.escape(key)}"=>"([^"]*)"')
                gdf.loc[mask, col] = gdf.loc[mask, "other_tags"].str.extract(pat, expand=False)
        return col

# --------------------------
# 1) LINES: filtrar por WHERE
# --------------------------
where_keys = ["aeroway","building","highway","railway"]
wcols_lines = {k: ensure_tag_col(lines, k) for k in where_keys}

aeroL = wcols_lines["aeroway"]
bldL  = wcols_lines["building"]
hwyL  = wcols_lines["highway"]
railL = wcols_lines["railway"]

m_lines = (
    lines[aeroL].notna() | (lines[bldL] == "aerodrome") |
    lines[hwyL].notna()  | lines[railL].notna()
)
lines_f = lines.loc[m_lines].copy()

# ahora sí, crear solo las columnas del SELECT sobre el subset
select_keys = ["aeroway","bridge","building","highway","layer","name","oneway",
               "railway","smoothness","surface","tunnel","width"]
selmap_lines = {k: ensure_tag_col(lines_f, k) for k in select_keys}
cols_out = ["osm_id"] + list(selmap_lines.values()) + ["geometry"]
lines_out = lines_f[cols_out].copy()
lines_out["source_layer"] = "lines"

print(f"LINES: {len(lines_out):,} seleccionadas de {len(lines):,}")

# ----------------------------------
# 2) MULTILINESTRINGS: mismo proceso
# ----------------------------------
# En esta capa casi todo viene en `other_tags`, así que usamos el helper.
wcols_mls = {k: ensure_tag_col(multilinestrings, k) for k in where_keys}

aeroM = wcols_mls["aeroway"]
bldM  = wcols_mls["building"]
hwyM  = wcols_mls["highway"]
railM = wcols_mls["railway"]

m_mls = (
    multilinestrings[aeroM].notna() | (multilinestrings[bldM] == "aerodrome") |
    multilinestrings[hwyM].notna()  | multilinestrings[railM].notna()
)
mls_f = multilinestrings.loc[m_mls].copy()

selmap_mls = {k: ensure_tag_col(mls_f, k) for k in select_keys}
cols_out_mls = ["osm_id"] + list(selmap_mls.values()) + ["geometry"]
mls_out = mls_f[cols_out_mls].copy()
mls_out["source_layer"] = "multilinestrings"

print(f"MULTILINESTRINGS: {len(mls_out):,} seleccionadas de {len(multilinestrings):,}")

# ----------------------------------
# 3) Unir en un único GeoDataFrame
# ----------------------------------
lines_like = pd.concat([lines_out, mls_out], ignore_index=True)
lines_like = gpd.GeoDataFrame(lines_like, geometry="geometry", crs=lines.crs)

print(f"TOTAL líneas (lines + multilinestrings): {len(lines_like):,}")
lines_like.head()

# OPTIMIZACIÓN: Liberar variables temporales que ya no se necesitan
del where_keys, wcols_lines, aeroL, bldL, hwyL, railL, m_lines, lines_f, selmap_lines, cols_out
del wcols_mls, aeroM, bldM, hwyM, railM, m_mls, mls_f, selmap_mls, cols_out_mls, lines_out, mls_out
_ = gc.collect()

In [ ]:
print(list(lines_like.columns))

In [ ]:
print(lines_like.notna().sum().sort_values(ascending=False))

In [ ]:
# lines_like.to_csv('lines_like.csv', index=False)  # Guarda sin la columna de índice

### Output Areas (OA)

In [ ]:
# 1) Cargar la DPA 2023
dpa_path = cfg.require(cfg.DPA_COMUNAS_SHP, "la cartografía DPA 2023", nivel=1)
comunas = gpd.read_file(dpa_path).to_crs(cfg.CRS_LATLON)
comunas.head()

In [ ]:
comunas.crs

In [ ]:
# 2) OA_ID estándar y columnas útiles
comunas["OA_ID"] = comunas["CUT_COM"].astype(str).str.zfill(5)

# 3) área geodésica (km²) + punto representativo
geod = Geod(ellps="WGS84")
def area_km2(geom):
    a_m2, _ = geod.geometry_area_perimeter(geom)
    return abs(a_m2) / 1e6

comunas["area_km2"] = comunas.geometry.apply(area_km2)
comunas["point_on_surface"] = comunas.geometry.representative_point()

output_areas = comunas[["OA_ID","REGION","PROVINCIA","COMUNA","area_km2","geometry"]].copy()
output_areas.head()

In [ ]:
# 4) Delimitación del caso de implementación.
#    La Provincia de Santiago acota el ámbito territorial del prototipo; no es
#    una restricción del pipeline, que opera sobre cualquier conjunto de
#    unidades administrativas. Ajuste este filtro para otro territorio.
output_areas = output_areas[output_areas["PROVINCIA"] == "Santiago"]
print(f"Output Areas seleccionadas: {len(output_areas)}")
output_areas.head()

In [ ]:
# 5) Guardar output_areas
output_areas.to_file(cfg.OUTPUT_AREAS_GEOJSON, driver="GeoJSON")
print(f"output_areas guardado: {cfg.OUTPUT_AREAS_GEOJSON} | unidades: {len(output_areas)}")

### Tessellation

In [ ]:
# --- Parámetros de teselación ---
TILE_SIZE_KM = cfg.TILE_SIZE_KM
TILE_SIZE_M = TILE_SIZE_KM * 1000
CRS_LATLON = cfg.CRS_LATLON
CRS_METERS = cfg.CRS_METRIC   # UTM 19S, adecuado para Chile continental centro-sur

# --- Cargar output_areas si no está en memoria ---
try:
    output_areas
except NameError:
    output_areas = gpd.read_file(cfg.OUTPUT_AREAS_GEOJSON).to_crs(CRS_LATLON)

# --- Construir ROI (unión de las unidades) en el CRS métrico ---
oa_m = output_areas.to_crs(CRS_METERS)
roi_geom = unary_union(oa_m.geometry).buffer(0)  # buffer(0) sanea geometrías

print(f"CRS métrico: {CRS_METERS} | área ROI (km²): {oa_m.area.sum()/1e6:.1f}")

In [ ]:
# --- bounds alineados a múltiplos del tamaño de celda ---
minx, miny, maxx, maxy = roi_geom.bounds
xmin = np.floor(minx / TILE_SIZE_M) * TILE_SIZE_M
ymin = np.floor(miny / TILE_SIZE_M) * TILE_SIZE_M
xmax = np.ceil (maxx / TILE_SIZE_M) * TILE_SIZE_M
ymax = np.ceil (maxy / TILE_SIZE_M) * TILE_SIZE_M

xs = np.arange(xmin, xmax, TILE_SIZE_M)  # columnas (O->E)
ys = np.arange(ymin, ymax, TILE_SIZE_M)  # filas (S->N)

# --- construir celdas y recortar al ROI ---
geoms, ids, rows, cols = [], [], [], []
for r, y in enumerate(ys[::-1]):                   # filas: N -> S
    for c, x in enumerate(xs):                     # cols: O -> E
        cell = box(x, y, x + TILE_SIZE_M, y + TILE_SIZE_M)
        if cell.intersects(roi_geom):
            geom = cell.intersection(roi_geom)
            if not geom.is_empty:
                geoms.append(geom)
                ids.append(f"T{r:03d}_{c:03d}")    # ID estable
                rows.append(r); cols.append(c)

tiles = gpd.GeoDataFrame(
    {"tile_id": ids, "row": rows, "col": cols, "geometry": geoms},
    crs=CRS_METERS
)
tiles["area_km2"] = tiles.area / 1e6

print(f"Tiles generados (antes de exportar): {len(tiles)}")
tiles.head(3)

In [ ]:
# --- Convertir a lat/lon y exportar ---
tiles_wgs = tiles.to_crs(CRS_LATLON)

tiles_wgs.to_file(cfg.TESSELLATION_GEOJSON, driver="GeoJSON")
print(f"tessellation.geojson guardado en: {cfg.TESSELLATION_GEOJSON}")

try:
    tiles_wgs.to_file(cfg.TESSELLATION_SHP)
    print(f"tessellation.shp guardado en: {cfg.TESSELLATION_SHP}")
except Exception as e:
    print("Shapefile opcional no guardado:", e)

print(
    f"N° tiles: {len(tiles_wgs)} | "
    f"Área total (km²): {tiles['area_km2'].sum():.1f} | "
    f"CRS: {tiles_wgs.crs}"
)

# Visualización de control
ax = output_areas.plot(facecolor="none", edgecolor="grey", linewidth=0.5)
tiles_wgs.boundary.plot(ax=ax, linewidth=0.7)
ax.set_title("Teselación vs Output Areas")
ax.set_axis_off()

El tamaño de tesela se determinó mediante el barrido del chunk siguiente, que
evalúa distintos valores frente a la distribución de unidades por tesela. El
criterio busca un compromiso entre número de teselas ocupadas y cantidad de
unidades por tesela, dado que la partición espacial se construye a nivel de
tesela. El valor adoptado se declara en `config.TILE_SIZE_KM`.

In [ ]:
# Explora varios tamaños (km) y mira distribución de OA por tile
CANDIDATES_KM = [8, 10, 12, 15, 20, 25, 30]

def build_tiles(tile_km):
    size_m = tile_km * 1000
    # bounds alineados
    minx, miny, maxx, maxy = roi_geom.bounds
    xmin = np.floor(minx / size_m) * size_m
    ymin = np.floor(miny / size_m) * size_m
    xmax = np.ceil(maxx / size_m) * size_m
    ymax = np.ceil(maxy / size_m) * size_m

    xs = np.arange(xmin, xmax, size_m)
    ys = np.arange(ymin, ymax, size_m)

    geoms, ids = [], []
    for r, y in enumerate(ys[::-1]):
        for c, x in enumerate(xs):
            cell = box(x, y, x + size_m, y + size_m)
            if cell.intersects(roi_geom):
                g = cell.intersection(roi_geom)
                if not g.is_empty:
                    geoms.append(g)
                    ids.append(f"T{r:03d}_{c:03d}")

    return gpd.GeoDataFrame({"tile_id": ids, "geometry": geoms}, crs=CRS_METERS)


def oa_per_tile(tiles_gdf):
    oa_pts = oa_m.copy()
    oa_pts["geometry"] = oa_pts.geometry.representative_point()
    j = gpd.sjoin(
        oa_pts[["OA_ID", "geometry"]],
        tiles_gdf[["tile_id", "geometry"]],
        how="left",
        predicate="within",
    )
    counts = j.groupby("tile_id").size()
    return counts


rows = []
for km in CANDIDATES_KM:
    t = build_tiles(km)
    c = oa_per_tile(t)
    rows.append({
        "tile_km": km,
        "tiles_totales": len(t),
        "tiles_con_OA": (c > 0).sum(),
        "OA_por_tile_min": int(c.min()),
        "OA_por_tile_p25": float(c.quantile(0.25)),
        "OA_por_tile_mediana": float(c.median()),
        "OA_por_tile_p75": float(c.quantile(0.75)),
        "OA_por_tile_max": int(c.max()),
    })

import pandas as pd
sweep = pd.DataFrame(rows).sort_values("tile_km")
print(sweep.to_string(index=False))

Bajo este criterio, la configuración adoptada en el caso de implementación fue
de 25 km por lado. En otro territorio, el barrido debe repetirse: el valor
óptimo depende de la extensión y del número de unidades administrativas.

### Construcción de features

In [ ]:
# A — Utilidades geodésicas y carga de Output Areas
os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"  # 0 = sin límite

oas = gpd.read_file(cfg.OUTPUT_AREAS_GEOJSON).to_crs(cfg.CRS_LATLON)
oas = oas[["OA_ID", "area_km2", "geometry"]].copy()

geod = Geod(ellps="WGS84")

def area_km2(geom):
    a_m2, _ = geod.geometry_area_perimeter(geom)
    return abs(a_m2) / 1e6

def length_km(geom):
    return geod.geometry_length(geom) / 1000.0

In [ ]:
# B — Conteo de POIs (points)
# Asegurar CRS
points_out = points_out.to_crs(4326)

# Join espacial (POIs dentro de OA)
pts = sjoin(points_out, oas[["OA_ID","geometry"]], how="inner", predicate="within")

# Diccionarios de categorías
AMEN_TRANSPORT = {"bus_station"}
AMEN_FOOD      = {"restaurant","fast_food","cafe","bar","pub"}
AMEN_HEALTH    = {"hospital","clinic","doctors","pharmacy","dentist"}
AMEN_EDU       = {"school","college","kindergarten","university","language_school"}
AMEN_RETAIL    = {"marketplace"}

# Helpers seguros (no crean columnas nuevas)
def has(col): return col in pts.columns
def B(s):     return s.fillna(False).astype(bool)          # fuerza Series booleana
def isin_col(col, values):
    return B(pts[col].isin(values)) if has(col) else pd.Series(False, index=pts.index)
def eq_col(col, value):
    return B(pts[col].eq(value))   if has(col) else pd.Series(False, index=pts.index)
def notna_col(col):
    return B(pts[col].notna())     if has(col) else pd.Series(False, index=pts.index)

# Máscaras (todas booleanas)
m_trans  = isin_col("amenity", AMEN_TRANSPORT) | eq_col("railway","station") \
           | notna_col("aeroway") | eq_col("highway","bus_stop")
m_food   = isin_col("amenity", AMEN_FOOD)
m_health = isin_col("amenity", AMEN_HEALTH) | notna_col("healthcare")
m_edu    = isin_col("amenity", AMEN_EDU)
m_retail = notna_col("shop") | isin_col("amenity", AMEN_RETAIL)

# Contadores por OA
def cnt(df, mask):
    return df.loc[mask].groupby("OA_ID").size().astype(float)

poi_trans  = cnt(pts, m_trans)
poi_food   = cnt(pts, m_food)
poi_health = cnt(pts, m_health)
poi_edu    = cnt(pts, m_edu)
poi_retail = cnt(pts, m_retail)

In [ ]:
# C — Conteo de “buildings” por categoría (multipolygons)

# 1) Asegurar CRS y representar cada polígono por un punto interior
multipolygons_out = multipolygons_out.to_crs(4326).copy()
mp_pts = multipolygons_out.copy()
mp_pts["geometry"] = mp_pts.geometry.representative_point()

# Nos quedamos solo con lo necesario para el conteo
keep_cols = ["osm_id","building","amenity","shop","geometry"]
mp_pts = mp_pts[[c for c in keep_cols if c in mp_pts.columns]]

# 2) Join espacial: asignar OA a cada polígono (vía su punto representativo)
mp = sjoin(mp_pts, oas[["OA_ID","geometry"]], how="inner", predicate="within")

# 3) Helpers seguros (sin crear columnas nuevas)
def has(col): return col in mp.columns
def B(s):     return s.fillna(False).astype(bool)
def isin_col(col, values): return B(mp[col].isin(values)) if has(col) else pd.Series(False, index=mp.index)
def eq_col(col, value):    return B(mp[col].eq(value))     if has(col) else pd.Series(False, index=mp.index)
def notna_col(col):        return B(mp[col].notna())       if has(col) else pd.Series(False, index=mp.index)

# 4) Familias temáticas de edificaciones
#    Cinco familias: transporte, alimentación, salud, educación y comercio.
AMEN_FOOD     = {"restaurant", "fast_food", "cafe", "bar", "pub"}
AMEN_HEALTH   = {"hospital", "clinic", "doctors", "pharmacy", "dentist"}
BLD_TRANSPORT = {"train_station", "aerodrome"}
BLD_EDU       = {"school", "university"}
BLD_RETAIL    = {"retail"}

# 5) Máscaras por familia (todas booleanas)
m_bld_trans  = isin_col("building", BLD_TRANSPORT)
m_bld_food   = isin_col("amenity", AMEN_FOOD)
m_bld_health = isin_col("amenity", AMEN_HEALTH) | eq_col("building","hospital")
m_bld_edu    = isin_col("building", BLD_EDU)
m_bld_retail = isin_col("building", BLD_RETAIL) | notna_col("shop")

# 6) Contadores por OA
def cnt(df, mask):
    return df.loc[mask].groupby("OA_ID").size().astype(float)

bld_trans  = cnt(mp, m_bld_trans)
bld_food   = cnt(mp, m_bld_food)
bld_health = cnt(mp, m_bld_health)
bld_edu    = cnt(mp, m_bld_edu)
bld_retail = cnt(mp, m_bld_retail)

print("Buildings por OA → listas las series: bld_trans, bld_food, bld_health, bld_edu, bld_retail")

In [ ]:
# D — Áreas de land use por OA (km²)

# 1) Subset de columnas y filtro mínimo
keep = ["geometry"]
for c in ["landuse","leisure","amenity"]:
    if c in multipolygons_out.columns:
        keep.append(c)

land = multipolygons_out[keep].to_crs(4326).copy()
mask_land = pd.Series(False, index=land.index)
if "landuse" in land.columns:
    mask_land |= land["landuse"].notna()
if "leisure" in land.columns:
    mask_land |= land["leisure"].eq("park")
if "amenity" in land.columns:
    mask_land |= land["amenity"].eq("grave_yard")

land = land[mask_land].copy()

# 2) Preselección espacial: qué polígonos intersectan con qué OA
cand = sjoin(land, oas[["OA_ID","geometry"]], how="inner", predicate="intersects")
# `cand.geometry` = geometría del polígono land; OA viene sin su geometría.
cand = cand.merge(
    oas[["OA_ID","geometry"]].rename(columns={"geometry":"geometry_oa"}),
    on="OA_ID", how="left"
)

# 3) Área geodésica de la intersección (km²)
def inter_area_km2(row):
    inter = row["geometry"].intersection(row["geometry_oa"])
    return area_km2(inter) if not inter.is_empty else 0.0

cand["area_km2"] = cand.apply(inter_area_km2, axis=1)

# 4) Definir clases (solo usando columnas existentes)
LU_RES = {"residential"}
LU_COM = {"commercial"}
LU_IND = {"industrial"}
LU_RET = {"retail"}
LU_NAT = {"forest","grass","meadow","scrub","heath","wood","wetland"}  # natural via landuse
# Extra: parques / cementerios como natural si existen
has_landuse = "landuse" in cand.columns
has_leisure = "leisure" in cand.columns
has_amenity = "amenity" in cand.columns

def B(series): return series.fillna(False).astype(bool)

m_res = B(cand["landuse"].isin(LU_RES)) if has_landuse else pd.Series(False, index=cand.index)
m_com = B(cand["landuse"].isin(LU_COM)) if has_landuse else pd.Series(False, index=cand.index)
m_ind = B(cand["landuse"].isin(LU_IND)) if has_landuse else pd.Series(False, index=cand.index)
m_ret = B(cand["landuse"].isin(LU_RET)) if has_landuse else pd.Series(False, index=cand.index)

m_nat = pd.Series(False, index=cand.index)
if has_landuse: m_nat |= B(cand["landuse"].isin(LU_NAT))
if has_leisure: m_nat |= B(cand["leisure"].eq("park"))
if has_amenity: m_nat |= B(cand["amenity"].eq("grave_yard"))

# 5) Sumar área por OA y por clase
def sum_area(mask):
    return cand.loc[mask, ["OA_ID","area_km2"]].groupby("OA_ID")["area_km2"].sum()

a_res = sum_area(m_res)
a_com = sum_area(m_com)
a_ind = sum_area(m_ind)
a_ret = sum_area(m_ret)
a_nat = sum_area(m_nat)

print("Áreas por OA listas → series: a_res, a_com, a_ind, a_ret, a_nat")

In [ ]:
# Chunk E — Longitud de red vial por OA (km)

# 1) Subset mínimo y CRS
roads = lines_like.to_crs(4326).copy()
if "highway" not in roads.columns:
    # No hay red vial; deja series vacías y sigue
    l_res = pd.Series(dtype=float)
    l_main = pd.Series(dtype=float)
    l_other = pd.Series(dtype=float)
    print("No se encontró columna 'highway' en lines_like. Series de longitudes vacías.")
else:
    roads = roads[roads["highway"].notna()][["geometry","highway"]].copy()

    # 2) Preselección espacial: qué tramos intersectan cada OA
    cand_r = sjoin(roads, oas[["OA_ID","geometry"]], how="inner", predicate="intersects")
    cand_r = cand_r.merge(
        oas[["OA_ID","geometry"]].rename(columns={"geometry":"geometry_oa"}),
        on="OA_ID", how="left"
    )

    # 3) Longitud geodésica de la intersección (km)
    def inter_len_km(row):
        inter = row["geometry"].intersection(row["geometry_oa"])
        return length_km(inter) if not inter.is_empty else 0.0

    cand_r["len_km"] = cand_r.apply(inter_len_km, axis=1)

    # 4) Clasificación de vías
    HWY_MAIN = {"motorway","trunk","primary"}
    HWY_RES  = {"residential","living_street"}
    # 'other' = resto

    # 5) Agregación por OA y clase
    def sum_len(mask):
        return cand_r.loc[mask, ["OA_ID","len_km"]].groupby("OA_ID")["len_km"].sum()

    s_res   = cand_r["highway"].isin(HWY_RES)
    s_main  = cand_r["highway"].isin(HWY_MAIN)
    s_other = ~s_res & ~s_main

    l_res   = sum_len(s_res)
    l_main  = sum_len(s_main)
    l_other = sum_len(s_other)

print("Longitudes por OA listas → series: l_res (res), l_main (main), l_other (other)")

In [ ]:
# F — Ensamblar features.csv (normalizado por área OA)

# 1) DataFrame base indexado por OA_ID
feat = oas[["OA_ID","area_km2"]].set_index("OA_ID").copy()

# 2) Agregar las 18 features (alineación por índice OA_ID)
# 2.a) Land use (5)
feat["lu_res_km2"] = a_res
feat["lu_com_km2"] = a_com
feat["lu_ind_km2"] = a_ind
feat["lu_ret_km2"] = a_ret
feat["lu_nat_km2"] = a_nat

# 2.b) Red vial (3)
feat["road_res_km"]   = l_res
feat["road_main_km"]  = l_main
feat["road_other_km"] = l_other

# 2.c) Facilities: POIs (5) + Buildings (5) = 10
feat["transport_pois"] = poi_trans
feat["food_pois"]      = poi_food
feat["health_pois"]    = poi_health
feat["education_pois"] = poi_edu
feat["retail_pois"]    = poi_retail

feat["transport_bld"]  = bld_trans
feat["food_bld"]       = bld_food
feat["health_bld"]     = bld_health
feat["education_bld"]  = bld_edu
feat["retail_bld"]     = bld_retail

# 3) Rellenar ausentes con 0
feat = feat.fillna(0.0)

# 4) Guardar versión “raw” (sin normalizar) para auditoría
features_raw = feat.reset_index()
features_raw.to_csv(cfg.FEATURES_RAW_CSV, index=False)
print("features_raw.csv guardado:", cfg.FEATURES_RAW_CSV)

# 5) Normalización por área_km2
#    - land use -> fracción de área (km²/km²)
#    - roads    -> densidad (km/km²)
#    - pois/bld -> densidad (cuentas/km²)
cols_to_norm = [c for c in feat.columns if c != "area_km2"]
den = np.where(feat["area_km2"].values > 0, feat["area_km2"].values, 1.0)  # evita división por 0
feat_norm = feat.copy()
feat_norm[cols_to_norm] = (feat_norm[cols_to_norm].T / den).T

# Renombrar a los identificadores esperados por el vector de atributos
feat_norm = feat_norm.rename(columns={
    "lu_res_km2":"lu_res", "lu_com_km2":"lu_com", "lu_ind_km2":"lu_ind", "lu_ret_km2":"lu_ret", "lu_nat_km2":"lu_nat",
    "road_res_km":"road_res", "road_main_km":"road_main", "road_other_km":"road_other",
})

# 6) Orden de columnas compatible con Deep Gravity
ordered_cols = ["area_km2",
                "lu_res", "lu_com", "lu_ind", "lu_ret", "lu_nat",
                "road_res", "road_main", "road_other",
                "transport_pois", "food_pois", "health_pois", "education_pois", "retail_pois",
                "transport_bld", "food_bld", "health_bld", "education_bld", "retail_bld"]
features = feat_norm.reindex(columns=ordered_cols)

# 7) Exportar CSV
features.reset_index().to_csv(cfg.FEATURES_CSV, index=False)
print("features.csv guardado:", cfg.FEATURES_CSV)
print("Shape:", features.shape)
display(features.reset_index().head())

Para finalizar el armado de features, añadimos la población como variable.

In [ ]:
# G — Población por comuna (Censo 2024)
xls_path = cfg.require(cfg.CENSO_XLSX, "el archivo de población del Censo 2024", nivel=1)

df = pd.read_excel(xls_path, sheet_name=cfg.CENSO_SHEET, header=cfg.CENSO_HEADER, dtype=str)
df.head()

In [ ]:
# Restringir al ámbito territorial del caso de implementación
df = df[df["Provincia"] == "Santiago"]
df = df[["Provincia", "Código comuna", "Comuna", "Población censada"]]
df.head()

In [ ]:
df.dtypes

In [ ]:
# 1) Normaliza nombres de columnas (quita espacios raros y NBSP)
df.columns = (df.columns.astype(str)
              .str.replace("\xa0", " ", regex=False)
              .str.strip()
              .str.replace(r"\s+", " ", regex=True))

# 2) Ubica columnas clave sin depender de mayúsculas/acentos perfectos
def pick(colname, candidates):
    # intenta coincidencia exacta y luego case-insensitive
    for c in candidates:
        if c in df.columns:
            return c
        m = [x for x in df.columns if x.lower() == c.lower()]
        if m:
            return m[0]
    raise KeyError(
    f"No se encontró ninguna de {candidates}. "
    f"Columnas disponibles: {df.columns.tolist()}"
    )

col_cod = pick("Código comuna", ["Código comuna","Codigo comuna"])
col_com = pick("Comuna",        ["Comuna"])
col_pop = pick("Población censada", ["Población censada","Poblacion censada","Población  censada"])

# 3) Limpia código de comuna → OA_ID de 5 dígitos
def norm_cod(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace(".0","").replace("\xa0","")
    s = re.sub(r"[^0-9]", "", s)         # deja solo dígitos
    return s.zfill(5) if s else np.nan

# 4) Limpia población → entero
def to_int(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("\xa0","").replace(" ", "")
    s = s.replace(".", "")               # miles con punto
    s = s.replace(",", ".")              # por si viene coma decimal
    try:
        return int(float(s))
    except Exception:
        return np.nan

tmp = df.copy()
tmp["OA_ID"]      = tmp[col_cod].map(norm_cod)
tmp["population"] = tmp[col_pop].map(to_int)

# 5) Filtra filas válidas (descarta totales tipo “País” si existieran)
mask = tmp["OA_ID"].notna() & tmp["population"].notna()
if col_com in tmp.columns:
    mask &= ~tmp[col_com].astype(str).str.strip().str.lower().isin(["país","pais"])

df_pop = (tmp.loc[mask, ["OA_ID","population"]]
            .groupby("OA_ID", as_index=False)["population"].sum())

print(f"Comunas con población válida: {len(df_pop):,}")
display(df_pop.head(10))

In [ ]:
features_with_pop = features.reset_index().merge(df_pop, on="OA_ID", how="left")
features_with_pop["population"] = features_with_pop["population"].fillna(0).astype(int)
features_with_pop["pop_density"] = np.where(
    features_with_pop["area_km2"] > 0,
    features_with_pop["population"] / features_with_pop["area_km2"],
    0.0
)

out_csv_pop = cfg.CHILE_DIR / "features_with_population.csv"
features_with_pop.to_csv(out_csv_pop, index=False)
print("features_with_population.csv guardado:", out_csv_pop)

In [ ]:
# ¿Hay OAs en features sin población?
oas_features = set(features.reset_index()["OA_ID"])
oas_pop      = set(df_pop["OA_ID"])
faltan = sorted(oas_features - oas_pop)
print("OAs sin población:", len(faltan))

In [ ]:
features_with_pop.head()

In [ ]:
# Ver las columnas que quedaron
print("Columnas finales:", features_with_pop.columns.tolist())

In [ ]:
# Consolidación de la infraestructura territorial en el directorio del dataset.
#
# pop_density se excluye del archivo final: el código de Deep Gravity incorpora
# log(población) por una ruta separada, de modo que mantener la densidad
# duplicaría la misma información dentro del vector de atributos.
f = features_with_pop.drop(columns=["pop_density"], errors="ignore")

# Controles de cobertura y tipos
assert set(f.OA_ID) == set(oas.OA_ID), "Los OA_ID no coinciden con la capa territorial"
numcols = [c for c in f.columns if c != "OA_ID"]
assert f[numcols].apply(pd.api.types.is_numeric_dtype).all(), "Hay columnas no numéricas"
assert np.isfinite(f[numcols].to_numpy()).all(), "Hay valores no finitos"

# Guardar en el directorio del dataset
f.to_csv(cfg.FEATURES_CSV, index=False)
oas.to_file(cfg.OUTPUT_AREAS_SHP)

# Teselación: usar la grilla en memoria o releerla del export previo
try:
    tiles_wgs.to_file(cfg.TESSELLATION_SHP)
    tiles_wgs.to_file(cfg.TESSELLATION_GEOJSON, driver="GeoJSON")
except NameError:
    tess = gpd.read_file(cfg.TESSELLATION_GEOJSON).to_crs(cfg.CRS_LATLON)
    tess.to_file(cfg.TESSELLATION_SHP)
    tess.to_file(cfg.TESSELLATION_GEOJSON, driver="GeoJSON")

print("features.csv, output_areas.* y tessellation.* listos en:", cfg.CHILE_DIR)

## Productos generados

Este cuaderno deja disponibles en `deepgravity/data/chile/`:

- `output_areas.*` — unidades territoriales del caso
- `tessellation.*` — grilla regular para la partición espacial
- `features.csv` — 18 atributos por unidad, normalizados por superficie, más población.

Todos derivan de fuentes públicas y forman parte de la entrega del repositorio. `output_areas.geojson` es además insumo de `preprocesamiento_trazas_telco.ipynb`, que lo utiliza para construir las transiciones entre comunas: quien regenere la infraestructura territorial desde cero debe ejecutar este cuaderno antes que aquel. La cadena continúa en `processed.ipynb`, que construye los objetos serializados que consume Deep Gravity.